# 模型参数

## 课程目标
* 理解 `max_tokens` 参数的作用
* 使用 `temperature` 参数控制模型响应
* 了解 `stop_sequence` 的用途

和往常一样，让我们首先导入 `anthropic` SDK 并加载我们的 API 密钥：

In [3]:
from dotenv import load_dotenv
from anthropic import Anthropic

#load environment variable
load_dotenv()

#automatically looks for an "ANTHROPIC_API_KEY" environment variable
client = Anthropic()

## 最大令牌数（Max tokens）

每次向 Claude 发出请求时，有三个必需参数必须包含：

* `model`
* `max_tokens`
* `messages`

到目前为止，我们一直在每个请求中使用 `max_tokens` 参数，但没有停下来讨论它是什么。

这是我们发出的第一个请求：

```py
our_first_message = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=500,
    messages=[
        {"role": "user", "content": "Hi there! Please write me a haiku about a pet chicken"}
    ]
)
```

那么 `max_tokens` 的目的是什么？

### 令牌（Tokens）
简而言之，`max_tokens` 控制 Claude 在其响应中生成的最大令牌数。在继续之前，让我们先停下来讨论一下令牌。

大多数大型语言模型并不以完整的单词"思考"，而是以一系列称为令牌的单词片段来处理。令牌是文本序列的小 building blocks，Claude 用它来处理、理解和生成文本。当我们向 Claude 提供提示时，该提示首先被转换为令牌并传递给模型。然后模型开始**一次生成一个令牌**的输出。

对于 Claude 来说，一个令牌大约代表 3.5 个英文字符，尽管确切的数字可能因使用的语言而异。

### 使用 `max_tokens`

`max_tokens` 参数允许我们设置 Claude 为我们生成的令牌数的上限。举个例子，假设我们要求 Claude 为我们写一首诗，并将 `max_tokens` 设置为 10。Claude 将开始为我们生成令牌，并在达到 10 个令牌时立即停止。这通常会导致截断或不完整的输出。让我们试试看！

In [2]:
truncated_response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=10,
    messages=[
        {"role": "user", "content": "Write me a poem"}
    ]
)
print(truncated_response.content[0].text)

Here is a poem for you:

The


这是我们从 Claude 收到的结果：

>Here is a poem for you:
>
>The

如果你运行上面的代码，你可能会得到一个同样被截断的不同结果。Claude 开始为我们写一首诗，然后在生成 10 个令牌后立即停止。


我们还可以检查响应 Message 对象上的 `stop_reason` 属性来查看模型停止生成的原因。在这种情况下，我们可以看到它的值是 "max_tokens"，这告诉我们模型停止生成是因为达到了我们的最大令牌限制！

In [26]:
truncated_response.stop_reason

'max_tokens'

当然，如果我们在 `max_tokens` 中使用更大的值来生成一首诗，我们可能会得到一首完整的诗：

In [27]:
longer_poem_response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=500,
    messages=[
        {"role": "user", "content": "Write me a poem"}
    ]
)
print(longer_poem_response.content[0].text)

Here is a poem for you:

Whispers of the Heart

In the quiet moments,
When the world fades away,
I hear the whispers of my heart -
Gentle words that gently sway.

They speak of dreams still unfurled,
Of love that shines like the sun,
Of passions yet to be explored,
Of all that's yet to be done.

These whispers, they guide my way,
Reminding me to pause and feel,
To listen closely to the soul,
And let its truths be revealed.

For in the silence, in the calm,
The heart's true voice can be heard,
Weaving a tapestry of hope,
With every softly spoken word.

So I will heed these whispers dear,
And let them be my faithful friend,
For in their song, I find my way,
To all my heart hopes to transcend.


这是 Claude 在 `max_tokens` 设置为 500 时生成的内容：

```
Here is a poem for you:

Whispers of the Wind

The wind whispers softly,
Caressing my face with care.
Its gentle touch, a fleeting breath,
Carries thoughts beyond compare.

Rustling leaves dance in rhythm,
Swaying to the breeze's song.
Enchanting melodies of nature,
Peaceful moments linger long.

The wind's embrace, a soothing balm,
Calms the restless soul within.
Embracing life's fleeting moments,
As the wind's sweet song begins.
```



如果我们查看这个响应的 `stop_reason`，会看到值为 "end_turn"，这是模型告诉我们它自然完成了生成的方式。它为我们写了一首诗，没有其他要说的了，所以停止了！

In [28]:
longer_poem_response.stop_reason

'end_turn'

需要注意的是，模型在生成内容时并不"知道" `max_tokens`。更改 `max_tokens` 不会改变 Claude 生成输出的方式，它只是给模型更多的空间继续生成（高 `max_tokens` 值）或截断输出（低 `max_tokens` 值）。

同样重要的是要知道，增加 `max_tokens` 并不能确保 Claude 实际生成特定数量的令牌。如果我们要求 Claude 写一个笑话并将 `max_tokens` 设置为 1000，我们几乎肯定会得到一个比 1000 个令牌短得多的响应。

In [4]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=1000,
    messages=[{"role": "user", "content": "Tell me a joke"}]
)

In [6]:
print(response.content[0].text)

Here's a classic dad joke for you:

Why don't scientists trust atoms? Because they make up everything!

How was that? I tried to keep it clean and mildly amusing. Let me know if you'd like to hear another joke.


In [7]:
print(response.usage.output_tokens)

55


在上面的例子中，我们要求 Claude "Tell me a joke"并将 `max_tokens` 设置为 1000。它生成了这个笑话：

```
Here's a classic dad joke for you:

Why don't scientists trust atoms? Because they make up everything!

How was that? I tried to keep it clean and mildly amusing. Let me know if you'd like to hear another joke.
```

生成的内容只有 55 个令牌。我们给 Claude 设置了 1000 个令牌的上限，但这并不意味着它会生成 1000 个令牌。

### 为什么要调整最大令牌数？
理解令牌对于使用 Claude 至关重要，特别是出于以下原因：

* **API 限制**：输入文本和生成响应中的令牌数量都计入 API 使用限制。每个 API 请求可以处理的最大令牌数有限制。了解令牌有助于你保持在 API 限制内并有效地管理使用。
* **性能**：Claude 生成的令牌数量直接影响 API 的处理时间和内存使用。更长的输入文本和更高的 max_tokens 值需要更多的计算资源。理解令牌有助于你优化 API 请求以获得更好的性能。
* **响应质量**：设置适当的 max_tokens 值可确保生成的响应具有足够的长度并包含必要的信息。如果 max_tokens 值太低，响应可能会被截断或不完整。尝试不同的 max_tokens 值可以帮助你找到特定用例的最佳平衡点。

让我们看看 Claude 生成的令牌数量如何影响性能。下面的函数三次要求 Claude 在两个角色之间生成一段很长的对话，每次使用不同的 `max_tokens` 值。然后打印出实际生成的令牌数和生成所用的时间。

In [4]:
import time
def compare_num_tokens_speed():
    token_counts = [100,1000,4096]
    task = """
        Create a long, detailed dialogue that is at least 5000 words long between two characters discussing the impact of social media on mental health. 
        The characters should have differing opinions and engage in a respectful thorough debate.
    """

    for num_tokens in token_counts:
        start_time = time.time()

        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=num_tokens,
            messages=[{"role": "user", "content": task}]
        )

        end_time = time.time()
        execution_time = end_time - start_time

        print(f"Number of tokens generated: {response.usage.output_tokens}")
        print(f"Execution Time: {execution_time:.2f} seconds\n")

In [5]:
compare_num_tokens_speed()

Number of tokens generated: 100
Execution Time: 1.51 seconds

Number of tokens generated: 1000
Execution Time: 8.33 seconds

Number of tokens generated: 3433
Execution Time: 28.80 seconds



如果你运行代码，你得到的确切值可能会有所不同，但以下是示例输出：

```
Number of tokens generated: 100
Execution Time: 1.51 seconds

Number of tokens generated: 1000
Execution Time: 8.33 seconds

Number of tokens generated: 3433
Execution Time: 28.80 seconds
```

正如你所看到的，**Claude 生成的令牌越多，所需时间就越长！**

For an even more obvious example, we asked Claude to repeat back a very long piece of text and used `max_tokens` to cut off the generation at various output sizes.  We repeated this 50 times for each size and calculated the average generation times.  As you can see, as the output size grows so does the time it takes! Take a look at the following plot:

![output_length.png](attachment:output_length.png)

## 停止序列（Stop sequences）

另一个我们尚未见到的重要参数是 `stop_sequence`，它允许我们为模型提供一组字符串，当这些字符串出现在生成的响应中时，会导致生成停止。它们本质上是一种告诉 Claude 的方式："如果你生成了这个序列，就停止生成任何其他内容！"

这是一个不包含 `stop_sequence` 的请求示例：

In [21]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=500,
    messages=[{"role": "user", "content": "Generate a JSON object representing a person with a name, email, and phone number ."}],
)
print(response.content[0].text)

Here's an example of a JSON object representing a person with a name, email, and phone number:

```json
{
  "name": "John Doe",
  "email": "johndoe@example.com",
  "phoneNumber": "123-456-7890"
}
```

In this example, the JSON object has three key-value pairs:

1. "name": The person's name, which is a string value of "John Doe".
2. "email": The person's email address, which is a string value of "johndoe@example.com".
3. "phoneNumber": The person's phone number, which is a string value of "123-456-7890".

You can modify the values to represent a different person with their own name, email, and phone number.


上面的代码要求 Claude 生成一个表示人物的 JSON 对象。以下是 Claude 生成的示例输出：

```
Here's an example of a JSON object representing a person with a name, email, and phone number:

{
  "name": "John Doe",
  "email": "johndoe@example.com",
  "phoneNumber": "123-456-7890"
}


In this example, the JSON object has three key-value pairs:

1. "name": The person's name, which is a string value of "John Doe".
2. "email": The person's email address, which is a string value of "johndoe@example.com".
3. "phoneNumber": The person's phone number, which is a string value of "123-456-7890".

You can modify the values to represent a different person with their own name, email, and phone number.
```

Claude 确实生成了请求的对象，但之后还附带了说明。如果我们想让 Claude 在生成 JSON 对象的右花括号 "}" 后立即停止生成，我们可以修改代码以包含 `stop_sequences` 参数。

In [23]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=500,
    messages=[{"role": "user", "content": "Generate a JSON object representing a person with a name, email, and phone number ."}],
    stop_sequences=["}"]
)
print(response.content[0].text)

Here's a JSON object representing a person with a name, email, and phone number:

{
  "name": "John Doe",
  "email": "john.doe@example.com",
  "phone": "555-1234"



模型生成了以下输出：

```
Here's a JSON object representing a person with a name, email, and phone number:
{
  "name": "John Doe",
  "email": "john.doe@example.com",
  "phone": "555-1234"

```
**重要提示：** 注意生成的输出**不包含** "}" 停止序列本身。如果我们想用它作为 JSON 进行解析，我们需要添加回闭合的 "}"。

当我们从 Claude 收到响应时，我们可以通过检查 `stop_reason` 属性来查看模型停止生成文本的原因。如下所示，上一个响应因 'stop_sequence' 而停止，这意味着模型生成了我们提供的停止序列之一并立即停止。

In [24]:
response.stop_reason

'stop_sequence'

我们还可以查看响应上的 `stop_sequence` 属性，以检查是哪个特定的 stop_sequence 导致模型停止生成：

In [25]:
response.stop_sequence

'}'

我们可以提供多个停止序列。如果我们提供多个，一旦模型遇到任何一个停止序列，它就会停止生成。响应 Message 上的 `stop_sequence` 属性会告诉我们遇到了哪个确切的 `stop_sequence`。

下面的函数要求 Claude 写一首诗，如果它生成字母 "b" 或 "c" 就停止。它运行三次：

In [37]:
def generate_random_letters_3_times():
    for i in range(3):
        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=500,
            messages=[{"role": "user", "content": "generate a poem"}],
            stop_sequences=["b", "c"]
        )
        print(f"Response {i+1} stopped because {response.stop_reason}.  The stop sequence was {response.stop_sequence}")

In [38]:
generate_random_letters_3_times()

Response 1 stopped because stop_sequence.  The stop sequence was c
Response 2 stopped because stop_sequence.  The stop sequence was b
Response 3 stopped because stop_sequence.  The stop sequence was b


以下是示例输出：

```
Response 1 stopped because stop_sequence.  The stop sequence was c
Response 2 stopped because stop_sequence.  The stop sequence was b
Response 3 stopped because stop_sequence.  The stop sequence was b
```

第一次，Claude 停止写诗是因为它生成了字母 "c"。接下来两次，它停止是因为生成了字母 "b"。你会这样做吗？可能不会！

## 温度（Temperature）

`temperature` 参数用于控制生成响应的"随机性"和"创造性"。它的范围是 0 到 1，较高的值会导致更多样化和不可预测的响应，措辞也会有变化。较低的温度会产生更确定性的输出，坚持最可能的措辞和答案。**温度的默认值为 1**。

在生成文本时，Claude 预测下一个令牌（单词或子单词）的概率分布。温度参数用于在采样下一个令牌之前操纵这个概率分布。如果温度低（接近 0.0），概率分布变得更加尖锐，高概率被分配给最可能的令牌。这使得模型更具确定性，专注于最可能或"安全"的选择。如果温度高（接近 1.0），概率分布变得更加平坦，不太可能令牌的概率也会增加。这使得模型更加随机和探索性，允许更多样化和创造性的输出。

请参阅此图表以获取温度影响的直观表示：

![temperature.png](attachment:temperature.png)


为什么要更改温度？

**对于分析性任务，使用接近 0.0 的温度；对于创造性和生成性任务，使用接近 1.0 的温度。**



让我们做一个快速演示。看看下面的函数。使用温度为 0 和温度为 1，我们向 Claude 发出三次请求，要求它"为一个外星星球想一个名字。只用一个词回复。"

In [47]:
def demonstrate_temperature():
    temperatures = [0, 1]
    for temperature in temperatures:
        print(f"Prompting Claude three times with temperature of {temperature}")
        print("================")
        for i in range(3):
            response = client.messages.create(
                model="claude-3-haiku-20240307",
                max_tokens=100,
                messages=[{"role": "user", "content": "Come up with a name for an alien planet. Respond with a single word."}],
                temperature=temperature
            )
            print(f"Response {i+1}: {response.content[0].text}")
        

In [48]:
demonstrate_temperature()

Prompting Claude three times with temperature of 0
Response 1: Xendor.
Response 2: Xendor.
Response 3: Xendor.
Prompting Claude three times with temperature of 1
Response 1: Xyron.
Response 2: Xandar.
Response 3: Zyrcon.


这是运行上述函数的结果（你的具体结果可能有所不同）：

```
Prompting Claude three times with temperature of 0
================
Response 1: Xendor.
Response 2: Xendor.
Response 3: Xendor.
Prompting Claude three times with temperature of 1
================
Response 1: Xyron.
Response 2: Xandar.
Response 3: Zyrcon.
```

注意，当温度为 0 时，三次响应都是相同的。请注意，即使温度为 0.0，结果也不会完全确定性。然而，与温度为 1 时的结果相比，差异很明显。每次响应都是一个完全不同的外星星球名称。

Below is a chart that illustrates the impact temperature can have on Claude's outputs.  Using this prompt, "Pick any animal in the world. Respond with only a single word: the name of the animal," we queried Claude 100 times with a temperature of 0.  We then did it 100 more times, but with a temperature of 1.  The plot below shows the frequencies of each animal response Claude came up with.

![temperature_plot.png](attachment:temperature_plot.png)

As you can see, with a temperature of 0, Claude responded with "Giraffe" every single time. Please remember that a temperature of 0 does not guarantee deterministic results, but it does make Claude much more likely to respond with similar content each time.  With a temperature of 1, Claude still chose giraffe more than half the time, but the responses also include many other types of animals!

## 系统提示词（System prompt）

`system_prompt` 是一个可选参数，你可以在向 Claude 发送消息时包含它。它通过给 Claude 提供高级指令、定义其角色或提供应影响其响应的背景信息来为对话奠定基础。

关于 system_prompt 的关键点：

* 它是可选的，但对于设置对话的语气和上下文很有用。
* 它在对话级别应用，影响该交换中 Claude 的所有响应。
* 它可以帮助引导 Claude 的行为，而无需在每条用户消息中包含指令。

请注意，在大多数情况下，只有语气、上下文和角色内容应该放在 system prompt 内部。详细指令、外部输入内容（如文档）和示例应该放在第一个 `User` 回合中以获得更好的结果。你不需要在每个后续的 `User` 回合中重复此操作。

让我们试试看：

In [4]:
message = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=1000,
    system="You are a helpful foreign language tutor that always responds in French.",
    messages=[
        {"role": "user", "content": "Hey there, how are you?!"}
    ]
)

print(message.content[0].text)

Bonjour ! Je suis ravi de vous rencontrer. Comment allez-vous aujourd'hui ?


***

## 练习

编写一个名为 `generate_questions` 的函数，实现以下功能：
* 接受两个参数：`topic` 和 `num_questions`
* 生成关于所提供 `topic` 的 `num_questions` 个发人深省的问题，作为编号列表
* 打印生成的问题

例如，调用 `generate_questions(topic="free will", num_questions=3)` 可能产生以下输出：


> 1. To what extent do our decisions and actions truly originate from our own free will, rather than being shaped by factors beyond our control, such as our genes, upbringing, and societal influences?
> 2. If our decisions are ultimately the result of a complex interplay of biological, psychological, and environmental factors, does that mean we lack the ability to make authentic, autonomous choices, or is free will compatible with determinism?
> 3. What are the ethical and philosophical implications of embracing or rejecting the concept of free will? How might our views on free will impact our notions of moral responsibility, punishment, and the nature of the human condition?


在你的实现中，请使用以下参数：
* `max_tokens` 将响应限制在 1000 个令牌以内
* `system` 提供一个系统提示词，告诉模型它是特定 `topic` 的专家，应该生成一个编号列表。
* `stop_sequences` 确保模型在生成正确数量的问题后停止。（如果我们要求 3 个问题，我们希望确保模型在生成 "4." 后立即停止。如果我们要求 5 个问题，我们希望确保模型在生成 "6." 后立即停止。）

#### 参考答案

In [1]:
def generate_questions(topic, num_questions=3):
    response = client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=500,
        system=f"You are an expert on {topic}. Generate thought-provoking questions about this topic.",
        messages=[
            {"role": "user", "content": f"Generate {num_questions} questions about {topic} as a numbered list."}
        ],
        stop_sequences=[f"{num_questions+1}."]
    )
    print(response.content[0].text)

In [5]:
generate_questions(topic="free will", num_questions=3)

Here are three thought-provoking questions about free will:

1. To what extent do our decisions and actions truly originate from our own free will, rather than being shaped by factors beyond our control, such as our genes, upbringing, and societal influences?

2. If our decisions are ultimately the result of a complex interplay of biological, psychological, and environmental factors, does that mean we lack the ability to make authentic, autonomous choices, or is free will compatible with determinism?

3. What are the ethical and philosophical implications of embracing or rejecting the concept of free will? How might our views on free will impact our notions of moral responsibility, punishment, and the nature of the human condition?


***